<a href="https://colab.research.google.com/github/tejalkv0/DL-project/blob/main/train_image_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2: Deep Learning — Image Classification with Transfer Learning

**Goal:** a small deep learning image classifier in PyTorch on a CIFAR-10 subset.

| Requirement | How it is met |
|---|---|
| Pretrained model / transfer learning | ImageNet-pretrained **ResNet-18**, new classification head, two-stage fine-tuning |
| Data augmentation | Random crop, horizontal flip, rotation, colour jitter |
| Training curves + evaluation metrics | Loss/accuracy curves, accuracy, precision/recall/F1, confusion matrix |
| Saved model + inference instructions | `cifar10_resnet18.pth` + `predict.py` + README |

**Run on Colab:** Runtime → Change runtime type → GPU (T4), then Runtime → Run all.

## 1. Setup

In [1]:
import json, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__)

Device: cuda | torch 2.11.0+cu130


## 2. Data: CIFAR-10 subset
To keep training fast, we use a **stratified subset** (equal images per class):
600 train, 100 validation, 200 test per class, giving 6,000 / 1,000 / 2,000 images.
Images are upscaled from 32×32 to 128×128 so the pretrained ResNet sees more spatial detail.

In [ ]:
IMG_SIZE = 128
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats (needed for the pretrained model)
TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS = 600, 100, 200
BATCH_SIZE = 64

train_tf = transforms.Compose([                      # DATA AUGMENTATION (training only)
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=8),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

root = "./data"
train_aug  = datasets.CIFAR10(root, train=True,  download=True, transform=train_tf)
train_eval = datasets.CIFAR10(root, train=True,  download=True, transform=eval_tf)   # same images, no augmentation
test_full  = datasets.CIFAR10(root, train=False, download=True, transform=eval_tf)
CLASSES = train_aug.classes
print(CLASSES)

def stratified_split(targets, sizes, seed=SEED):
    rng = np.random.RandomState(seed)
    targets = np.array(targets)
    out = [[] for _ in sizes]
    for c in range(len(CLASSES)):
        idx = rng.permutation(np.where(targets == c)[0])
        start = 0
        for k, n in enumerate(sizes):
            out[k].extend(idx[start:start+n].tolist()); start += n
    return out

train_idx, val_idx = stratified_split(train_aug.targets, [TRAIN_PER_CLASS, VAL_PER_CLASS])
(test_idx,) = stratified_split(test_full.targets, [TEST_PER_CLASS])

train_loader = DataLoader(Subset(train_aug,  train_idx), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(Subset(train_eval, val_idx),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(Subset(test_full,  test_idx),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

 18%|█▊        | 30.2M/170M [09:19<42:21, 55.2kB/s]

### Augmentation preview

In [ ]:
inv = transforms.Normalize([-m/s for m, s in zip(MEAN, STD)], [1/s for s in STD])
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
base = Subset(datasets.CIFAR10(root, train=True, transform=None), train_idx[:6])
for i in range(6):
    img, label = base[i]
    axes[0, i].imshow(img.resize((IMG_SIZE, IMG_SIZE))); axes[0, i].set_title(CLASSES[label]); axes[0, i].axis("off")
    aug = inv(train_tf(img)).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[1, i].imshow(aug); axes[1, i].set_title("augmented"); axes[1, i].axis("off")
plt.tight_layout(); plt.savefig("augmentation_preview.png", dpi=120); plt.show()

## 3. Model: ResNet-18 with transfer learning
Two-stage fine-tuning:
1. **Stage 1 (head only):** freeze the pretrained backbone, train only the new classifier layer.
2. **Stage 2 (fine-tune):** unfreeze the last residual block (`layer4`) and train with a lower learning rate.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(CLASSES)))
model = model.to(device)

def set_trainable(model, stage):
    for p in model.parameters(): p.requires_grad = False
    for p in model.fc.parameters(): p.requires_grad = True
    if stage == 2:
        for p in model.layer4.parameters(): p.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Stage {stage}: trainable parameters = {n:,}")

criterion = nn.CrossEntropyLoss()
use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_epoch(loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * x.size(0)
            preds = out.argmax(1)
            correct += (preds == y).sum().item(); n += x.size(0)
            all_preds.append(preds.cpu()); all_labels.append(y.cpu())
    return total_loss / n, correct / n, torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

## 4. Training

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
STAGE1_EPOCHS, STAGE2_EPOCHS = 4, 6
best_val, best_state = 0.0, None

def train_stage(stage, epochs, lr):
    global best_val, best_state
    set_trainable(model, stage)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta, _, _ = run_epoch(train_loader, optimizer)
        vl, va, _, _ = run_epoch(val_loader)
        scheduler.step()
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        if va > best_val:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"[stage {stage}] epoch {ep}/{epochs}  train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}  ({time.time()-t0:.0f}s)")

train_stage(1, STAGE1_EPOCHS, lr=1e-3)
train_stage(2, STAGE2_EPOCHS, lr=3e-4)
model.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_val:.4f}")

## 5. Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, history["train_loss"], label="train"); ax[0].plot(epochs, history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(epochs, history["train_acc"], label="train"); ax[1].plot(epochs, history["val_acc"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
for a in ax: a.axvline(STAGE1_EPOCHS + 0.5, color="gray", ls="--", lw=1)   # stage 1 -> stage 2 (fine-tuning begins)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 6. Evaluation on the held-out test set

In [ ]:
test_loss, test_acc, preds, labels = run_epoch(test_loader)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}\n")
report = classification_report(labels, preds, target_names=CLASSES, digits=3, output_dict=True)
print(classification_report(labels, preds, target_names=CLASSES, digits=3))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues"); plt.colorbar(im)
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test set)")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

metrics = {"test_accuracy": test_acc, "test_loss": test_loss, "best_val_accuracy": best_val,
           "macro_precision": report["macro avg"]["precision"], "macro_recall": report["macro avg"]["recall"],
           "macro_f1": report["macro avg"]["f1-score"], "per_class": {c: report[c] for c in CLASSES}}
json.dump(metrics, open("metrics.json", "w"), indent=2)

### Sample predictions

In [ ]:
model.eval()
x, y = next(iter(test_loader))
with torch.no_grad(): p = model(x.to(device)).argmax(1).cpu()
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, a in enumerate(axes.flat):
    a.imshow(inv(x[i]).clamp(0, 1).permute(1, 2, 0).numpy()); a.axis("off")
    a.set_title(f"{CLASSES[p[i]]}", color="green" if p[i] == y[i] else "red", fontsize=10)
plt.suptitle("Predictions (green = correct, red = wrong)"); plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=120); plt.show()

## 7. Save the model

In [ ]:
CHECKPOINT = "cifar10_resnet18.pth"
torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "classes": CLASSES, "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
            "arch": "resnet18", "test_accuracy": test_acc}, CHECKPOINT)
print("Saved", CHECKPOINT)

# Sanity check: reload from disk and confirm identical predictions
ckpt = torch.load(CHECKPOINT, map_location="cpu")
m2 = models.resnet18(weights=None)
m2.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, len(ckpt["classes"])))
m2.load_state_dict(ckpt["state_dict"]); m2.eval()
with torch.no_grad(): p2 = m2(x[:12]).argmax(1)
print("Reload matches:", bool((p2 == p[:12]).all()))

## 8. Inference on a single image
Run `predict.py` (see README) from the command line, or use the function below.

In [ ]:
from PIL import Image
def predict_image(path, topk=3):
    img = Image.open(path).convert("RGB")
    t = eval_tf(img).unsqueeze(0)
    with torch.no_grad(): probs = torch.softmax(m2(t), 1)[0]
    top = probs.topk(topk)
    return [(ckpt["classes"][i], float(s)) for s, i in zip(top.values, top.indices)]

# Demo: save one test image and predict it
img, lab = datasets.CIFAR10(root, train=False)[0]; img.save("demo.png")
print("True label:", CLASSES[lab]); print(predict_image("demo.png"))

## 9. Results summary and discussion
*(Fill in the numbers from your own run — they are printed above and saved to `metrics.json`.)*

- **Test accuracy:** ___  |  **Macro F1:** ___
- Stage 1 (frozen backbone) already gives strong accuracy because ImageNet features transfer well; Stage 2 fine-tuning of `layer4` adds a few more points.
- Typical confusions are visually similar classes (cat ↔ dog, automobile ↔ truck) — see the confusion matrix.
- Augmentation keeps the train/validation gap small despite the small training set.
- **Possible improvements:** train on the full 50k images, larger resolution, unfreeze more layers, try EfficientNet/ConvNeXt.